### Step:1 Importing all the libraries 


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

### Step 2: Loading the datasets 


In [ ]:
df = pd.read_csv("earthquake.csv")

### Step 3: Look at the dataset first

In [ ]:
print("Shape of dataset (rows, columns):", df.shape)
print("\nFirst 5 rows of dataset:")
print(df.head())

print("\nColumn names:")
print(df.columns.tolist())

print("\nMissing values in each column:")
print(df.isnull().sum())

### STEP 4: DATA PREPROCESSING
#### latitude, longitude and depth (these describe WHERE the earthquake happened) we also keep magnitude to study the clusters later

In [ ]:
columns_needed = ["latitude", "longitude", "depth", "mag"]
df_cluster = df[columns_needed].copy()

### Remove rows that have missing values (simple way to handle missing data)

In [ ]:
print("\nRows before removing missing values:", df_cluster.shape[0])
df_cluster = df_cluster.dropna()
print("Rows after removing missing values:", df_cluster.shape[0])

### Remove duplicate rows if any

In [ ]:
df_cluster = df_cluster.drop_duplicates()
print("Rows after removing duplicates:", df_cluster.shape[0])

### STEP 5: FEATURE ENGINEERING

In [ ]:
features = df_cluster[["latitude", "longitude", "depth"]]

scaler = StandardScaler()
features_scaled = scaler.fit_transform(features)

print("\nFeatures before scaling (first 5 rows):")
print(features.head())

print("\nFeatures after scaling (first 5 rows):")
print(features_scaled[:5])

### STEP 6: FIND THE BEST NUMBER OF CLUSTERS (K)
###  We use the Elbow Method to choose a good value of K
### We try K = 2 to 10 and check the "inertia" (how tight the clusters are)


In [ ]:

inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans_test.fit(features_scaled)
    inertia_values.append(kmeans_test.inertia_)

print("\nInertia values for K = 2 to 10:")
for k, inertia in zip(k_range, inertia_values):
    print(f"K = {k} -> Inertia = {inertia:.2f}")



### We also check the Silhouette Score for a few K values
### Silhouette score tells us how well separated the clusters are (closer to 1 is better)

In [ ]:
print("\nSilhouette scores for K = 2 to 8:")
silhouette_scores = []
for k in range(2, 9):
    kmeans_test = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels_test = kmeans_test.fit_predict(features_scaled)
    score = silhouette_score(features_scaled, labels_test)
    silhouette_scores.append(score)
    print(f"K = {k} -> Silhouette Score = {score:.4f}")

### Based on the elbow method and silhouette score, we choose our final K
### (After checking both results, K = 5 was selected for this project)

In [ ]:
best_k = 5

### STEP 7: TRAIN THE FINAL K-MEANS MODEL

In [ ]:
kmeans_model = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_cluster["cluster"] = kmeans_model.fit_predict(features_scaled)

print(f"\nFinal model trained with K = {best_k}")
print("\nNumber of earthquakes in each cluster:")
print(df_cluster["cluster"].value_counts())


### STEP 8: EVALUATE THE MODEL

In [ ]:
final_silhouette = silhouette_score(features_scaled, df_cluster["cluster"])
print(f"\nFinal Silhouette Score for K = {best_k}: {final_silhouette:.4f}")

# Show the center of each cluster (in original scale, not scaled values)
cluster_centers_scaled = kmeans_model.cluster_centers_
cluster_centers_original = scaler.inverse_transform(cluster_centers_scaled)

print("\nCluster centers (latitude, longitude, depth):")
centers_df = pd.DataFrame(cluster_centers_original, columns=["latitude", "longitude", "depth"])
print(centers_df)


### STEP 9: ANALYZE EACH CLUSTER (FINDINGS)

In [ ]:
print("\nAverage magnitude and depth per cluster:")
cluster_summary = df_cluster.groupby("cluster")[["mag", "depth"]].mean()
print(cluster_summary)

print("\nMax magnitude found in each cluster:")
cluster_max_mag = df_cluster.groupby("cluster")["mag"].max()
print(cluster_max_mag)


### STEP 10: SAVE THE RESULTS

In [ ]:
df_cluster.to_csv("earthquake_with_clusters.csv", index=False)
print("\nClustered data saved to earthquake_with_clusters.csv")

print("\n=== K-MEANS CLUSTERING COMPLETE ===")